# Colab Setup

Mount Google Drive and pull the latest repo changes before running the experiment cells.

In [1]:
!pip install tensordict
!pip install torchRL

In [2]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
except Exception as exc:
    print('Google Drive mount skipped:', exc)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted


In [3]:
import subprocess
from pathlib import Path

REPO_PATH = Path('/content/drive/MyDrive/repos/evolutionary-ai-battle')

if REPO_PATH.exists():
    result = subprocess.run(['git', 'pull'], cwd=REPO_PATH, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
else:
    print(f'Repo path not found: {REPO_PATH}')

Already up to date.



In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = REPO_PATH if 'REPO_PATH' in globals() and REPO_PATH.exists() else Path.cwd()
EXPERIMENT_ROOT = PROJECT_ROOT / 'experiment' if PROJECT_ROOT.name != 'experiment' else PROJECT_ROOT

for path in (PROJECT_ROOT, EXPERIMENT_ROOT):
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
    print('autoreload enabled')
except Exception as exc:
    print('autoreload skipped:', exc)

try:
    import json
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('DEVICE:', DEVICE)
    if DEVICE == 'cuda':
        print('GPU:', torch.cuda.get_device_name(0))
        print('CUDA version:', torch.version.cuda)
    else:
        print('No GPU is visible to this runtime. In Colab, use Runtime > Change runtime type > GPU, then rerun setup.')
except Exception as exc:
    DEVICE = 'cpu'
    print('torch device check skipped:', exc)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('EXPERIMENT_ROOT:', EXPERIMENT_ROOT)

# Notebook training knobs. Adjust these in Colab before running the PPO cell.
CONFIG_PATH = EXPERIMENT_ROOT / 'configs' / 'ppo_smoke.yaml'
RUN_DIR = EXPERIMENT_ROOT / 'runs'
NOTEBOOK_SMOKE = True
NOTEBOOK_TOTAL_STEPS = 256
NOTEBOOK_ROLLOUT_STEPS = 64
NOTEBOOK_MAX_EPISODE_STEPS = 50
NOTEBOOK_HIDDEN_DIM = 64
NOTEBOOK_SELECTION_EVAL_EPISODES = 2
print('CONFIG_PATH:', CONFIG_PATH)
print('RUN_DIR:', RUN_DIR)

autoreload skipped: No module named 'imp'
DEVICE: cuda
GPU: Tesla T4
CUDA version: 12.8
PROJECT_ROOT: /content/drive/MyDrive/repos/evolutionary-ai-battle
EXPERIMENT_ROOT: /content/drive/MyDrive/repos/evolutionary-ai-battle/experiment


# CPC TorchRL Scaffold

This notebook is the initiating experiment entry point. Core logic lives in Python modules; the notebook demonstrates the toy CPC loop, the TorchRL wrapper, and PPO smoke training.

In [5]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else Path.cwd()
EXPERIMENT_ROOT = EXPERIMENT_ROOT if 'EXPERIMENT_ROOT' in globals() else PROJECT_ROOT / "experiment"

for path in (PROJECT_ROOT, EXPERIMENT_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from training.cpc_actions import action_space, decode_action, describe_action
from training.cpc_env import CPCEnv

print("experiment root:", EXPERIMENT_ROOT)
print("action space:", action_space())

experiment root: /content/drive/MyDrive/repos/evolutionary-ai-battle/experiment
action space: ActionSpace(move_bins=9, aim_bins=16, fire_bins=2)


## PR1: Toy CPC Loop

In [6]:
env = CPCEnv(seed=7, max_steps=8)
obs = env.reset()
move_and_fire = {"move": 6, "aim": 0, "fire": 1}

print(json.dumps(describe_action(move_and_fire), indent=2))
obs, reward, done, info = env.step(move_and_fire)
print("reward", reward, "done", done)
print("decoded", info["decoded_action"])

for _ in range(3):
    action = env.sample_action()
    obs, reward, done, info = env.step(action)
    print({"raw": action, "reward": round(reward, 3), "done": done})
    if done:
        break

print("metrics", json.dumps(env.metrics.summary(), indent=2))
print("trajectory step", json.dumps(env.export_trajectory()["steps"][0], indent=2))

{
  "rawAction": {
    "move": 6,
    "aim": 0,
    "fire": 1
  },
  "moveLabel": "up_right",
  "decoded": {
    "moveX": 0.7071067811865475,
    "moveY": -0.7071067811865475,
    "aimX": 1.0,
    "aimY": 0.0,
    "fire": 1
  }
}
reward 0.02 done False
decoded {'moveX': 0.7071067811865475, 'moveY': -0.7071067811865475, 'aimX': 1.0, 'aimY': 0.0, 'fire': 1}
{'raw': {'move': 5, 'aim': 4, 'fire': 1}, 'reward': 0.02, 'done': False}
{'raw': {'move': 0, 'aim': 2, 'fire': 0}, 'reward': 0.02, 'done': False}
{'raw': {'move': 5, 'aim': 1, 'fire': 0}, 'reward': 0.02, 'done': False}
metrics {
  "avg_ally_distance": 162.45319070603566,
  "isolation_rate": 0.0,
  "teammate_under_pressure_response": 0.0,
  "damage_dealt": 0.0,
  "damage_taken": 0.0
}
trajectory step {
  "step": 1,
  "actorId": "self",
  "action": {
    "moveX": 0.7071067811865475,
    "moveY": -0.7071067811865475,
    "aimX": 1.0,
    "aimY": 0.0,
    "fire": 1
  },
  "rawAction": {
    "move": 6,
    "aim": 0,
    "fire": 1
  },
  "s

## PR2: TorchRL EnvBase / TensorDict Wrapper

In [7]:
try:
    import torch
    from training.torchrl_env import TorchRLCPCEnv
    from training.torchrl_specs import import_check_env_specs

    torchrl_env = TorchRLCPCEnv(seed=11, max_steps=8, device=DEVICE)
    print("observation_spec")
    print(torchrl_env.observation_spec)
    print("action_spec")
    print(torchrl_env.action_spec)

    td = torchrl_env.reset()
    td.update(torchrl_env.action_spec.rand())
    stepped = torchrl_env.step(td)
    print("sample reward", stepped["next", "reward"], "done", stepped["next", "done"])

    move_fire_td = torchrl_env.reset()
    move_fire_td["move"] = torch.tensor(6, dtype=torch.int64)
    move_fire_td["aim"] = torch.tensor(0, dtype=torch.int64)
    move_fire_td["fire"] = torch.tensor(1, dtype=torch.int64)
    move_fire_next = torchrl_env.step(move_fire_td)
    print("decoded engine action", move_fire_next["next", "decoded_action"])

    check_env_specs = import_check_env_specs()
    if check_env_specs is not None:
        check_env_specs(torchrl_env)
        print("check_env_specs passed")
except Exception as exc:
    print("TorchRL wrapper demo skipped:", exc)

observation_spec
Composite(
    self_hp: UnboundedContinuous(
        shape=torch.Size([1]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([1]), device=cuda:0, dtype=torch.float32, contiguous=True),
            high=Tensor(shape=torch.Size([1]), device=cuda:0, dtype=torch.float32, contiguous=True)),
        device=cuda:0,
        dtype=torch.float32,
        domain=continuous),
    ally_hp: UnboundedContinuous(
        shape=torch.Size([1]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([1]), device=cuda:0, dtype=torch.float32, contiguous=True),
            high=Tensor(shape=torch.Size([1]), device=cuda:0, dtype=torch.float32, contiguous=True)),
        device=cuda:0,
        dtype=torch.float32,
        domain=continuous),
    enemy_hp: UnboundedContinuous(
        shape=torch.Size([1]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([1]), device=cuda:0, dtype=torch.float32, contiguous=True),
            high=Tensor(s

# PR3: PPO Smoke Training

This section trains through the notebook, saves reward-selected checkpoints, evaluates `checkpoint_min_reward.pt`, and loads it through `PPOPolicyAgent`. Adjust the notebook training knobs in the setup cell before running this cell.

In [9]:
try:
    import torch
    from training.ppo_policy import MultiDiscreteActorCritic
    from training.train_ppo import PPOConfig, collect_rollout, load_config, train_ppo
    from training.eval_ppo import eval_checkpoint
    from training.torchrl_env import TorchRLCPCEnv
    from training.cpc_actions import decode_action
    from training.cpc_env import CPCEnv
    from policy_agent import PPOPolicyAgent
    from run_model_agents import run_two_agent_eval

    env = TorchRLCPCEnv(seed=5, max_steps=8, device=DEVICE)
    policy = MultiDiscreteActorCritic(hidden_dim=32).to(DEVICE)
    obs = env.reset()
    output = policy.sample_action(obs)
    print("sampled policy action", {k: int(v) for k, v in output.action.items()})
    print("log_prob", float(output.log_prob.reshape(-1)[0]), "value", float(output.value.reshape(-1)[0]))

    rollout = collect_rollout(env, policy, PPOConfig(device=DEVICE, rollout_steps=8, max_episode_steps=8, hidden_dim=32))
    print("rollout obs shape", rollout["observations"].shape)

    cfg = load_config(CONFIG_PATH, smoke=NOTEBOOK_SMOKE)
    cfg.device = DEVICE
    cfg.run_dir = str(RUN_DIR)
    cfg.total_steps = NOTEBOOK_TOTAL_STEPS
    cfg.rollout_steps = NOTEBOOK_ROLLOUT_STEPS
    cfg.max_episode_steps = NOTEBOOK_MAX_EPISODE_STEPS
    cfg.hidden_dim = NOTEBOOK_HIDDEN_DIM
    cfg.selection_eval_episodes = NOTEBOOK_SELECTION_EVAL_EPISODES
    print("training config")
    print(json.dumps(cfg.__dict__, indent=2))

    result = train_ppo(cfg)
    LAST_TRAIN_RESULT = result
    SELECTED_CHECKPOINT = result["checkpoint_min_reward"]
    LATEST_CHECKPOINT = result["checkpoint_latest"]
    print("train result")
    print(json.dumps(result, indent=2))
    print("checkpoint_latest", LATEST_CHECKPOINT)
    print("checkpoint_min_reward", SELECTED_CHECKPOINT)

    report = eval_checkpoint(SELECTED_CHECKPOINT, episodes=2, device="cpu", deterministic=True)
    LAST_EVAL_REPORT = report
    print("eval report")
    print(json.dumps(report, indent=2))

    model_agent = PPOPolicyAgent.from_checkpoint(SELECTED_CHECKPOINT, device="cpu")
    agent_obs = CPCEnv(seed=7, max_steps=8).reset()
    raw_action = model_agent.act(agent_obs, deterministic=True)
    print("model agent raw_action", raw_action)
    print("model agent decoded_action", decode_action(raw_action))

    try:
        run_two_agent_eval(SELECTED_CHECKPOINT, episodes=1, device="cpu", deterministic=True)
    except NotImplementedError as runner_exc:
        print("two-agent model eval blocked:", runner_exc)
except Exception as exc:
    print("PPO smoke demo skipped:", exc)

# TODO PR4: scenario GT evaluation, trajectory export, and richer CPC metric reports.

sampled policy action {'move': 5, 'aim': 10, 'fire': 1}
log_prob -5.589089870452881 value 0.0802709087729454
rollout obs shape torch.Size([8, 13])
train result
{
  "run_dir": "experiment/runs/ppo_smoke_20260618_041808",
  "checkpoint": "experiment/runs/ppo_smoke_20260618_041808/checkpoint.pt",
  "metrics_csv": "experiment/runs/ppo_smoke_20260618_041808/metrics.csv",
  "device": "cuda",
  "last_metrics": {
    "update": 2,
    "step": 32,
    "episodic_return_mean": 0.08499999810010195,
    "episode_length_mean": 8.0,
    "policy_loss": 0.0781550481915474,
    "value_loss": 0.029936574399471283,
    "entropy": 5.63474702835083,
    "approx_kl": 0.0032762885093688965,
    "clip_fraction": 0.0,
    "avg_ally_distance": 162.07078552246094,
    "isolation_rate": 0.0,
    "damage_dealt": 0.0,
    "damage_taken": 0.0
  }
}
eval report
{
  "mean_episode_return": -0.2900000028312206,
  "mean_episode_length": 8.0,
  "mean_metrics": {
    "avg_ally_distance": 100.79161071777344,
    "isolation_ra

# PR3 Acceptance Check

This runs the merge-gate checks for PPO smoke training correctness. It checks seed reproducibility, move+fire, raw/decoded action visibility, checkpoint loading, metrics CSV rows, and 10-episode eval robustness.

In [ ]:
try:
    from check_pr3_acceptance import check_forced_move_fire_action, run_acceptance

    status, forced = check_forced_move_fire_action(seed=123)
    print("forced action check:", status)
    print(json.dumps(forced, indent=2))

    report = run_acceptance(
        config=EXPERIMENT_ROOT / "configs" / "ppo_smoke.yaml",
        seed=123,
        eval_episodes=10,
        device=DEVICE,
    )
    print(json.dumps(report, indent=2))
except Exception as exc:
    print("PR3 acceptance check skipped:", exc)